In [1]:
## for vscode notebook , environment should be set to python3, and run the following command to install dependencies
# !pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [34]:
from llmcompressor.modifiers.quantization import GPTQModifier

In [35]:
import warnings
warnings.filterwarnings("ignore")

import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ['TOKENIZERS_PARALLELISM'] = 'false'



In [36]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [37]:
import os
# Replace with your actual Hugging Face token
# os.environ["HF_TOKEN"]
hf_var= os.getenv("HF_TOKEN")
# hf_var

In [38]:
MODEL_DIR = "../models/Qwen3-0.6B"
OUTPUT_DIR = "../models/Qwen3-0.6B-W4A16"

print(f"Base model:      {MODEL_DIR}")
# print(f"Quantized model: {OUTPUT_DIR}")

Base model:      ../models/Qwen3-0.6B


In [39]:
!hf download Qwen/Qwen3-0.6B --local-dir  {MODEL_DIR}

Ignored error while writing commit hash to /home/johnsonhk88/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/refs/main: [Errno 13] Permission denied: '/home/johnsonhk88/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/refs/main'.
Fetching 10 files: 100%|█████████████████████| 10/10 [00:00<00:00, 3283.73it/s]
Download complete: : 0.00B [00:00, ?B/s]              ✓ Downloaded
  path: /media/johnsonhk88/Big-Data-Disk7/llm-model-convert-compression-evalution/models/Qwen3-0.6B
Download complete: : 0.00B [00:00, ?B/s]


In [40]:
import os

In [41]:
!python --version

Python 3.12.13


A **recipe** tells LLM Compressor how to quantize. It's a list of modifiers: each one specifies an algorithm and settings we'll apply to the model.

### Available Modifiers

**Quantization Modifiers:**

| Modifier | Description |
|:--|:--|
| `GPTQModifier` | GPTQ algorithm: uses calibration data to find optimal quantization values |
| `AWQModifier` | Activation-Weighted Quantization preserves salient weights (the weights that matter most) |
| `AutoRoundModifier` | Intel's algorithm with learnable rounding/clipping |
| `QuantizationModifier` | Basic PTQ and QAT for simple use cases |

**Transform Modifiers** (for improving accuracy):

| Modifier | Description |
|:--|:--|
| `SmoothQuantModifier` | Smooths activations before quantization, often paired with GPTQ |
| `QuIPModifier` | Hadamard rotations to reduce outliers |
| `SpinQuantModifier` | SpinQuant-style rotations to even out weight distributions |

Modifiers can be chained: e.g. applying `SmoothQuantModifier` before `GPTQModifier` improves accuracy for W8A8 quantization.

### Quantization Schemes

The `scheme` parameter determines the bit-width for weights (W) and activations (A):

| Scheme | Weights | Activations | Quantized Layer Reduction | Quality Impact |
|:--|:--|:--|:--|:--|
| `W8A16` | 8-bit | 16-bit (FP16) | ~50% | Minimal |
| `W4A16` | 4-bit | 16-bit (FP16) | ~75% | Low–Moderate |
| `W8A8` | 8-bit | 8-bit | ~50% | Low |
| `W4A8` | 4-bit | 8-bit | ~75% | Moderate |

> **Note:** These reductions apply to the quantized layers only. The embedding and `lm_head` layers are kept at full precision, so total model size reduction depends on how large those layers are relative to the rest. For small models (~0.6B), expect ~40–50% total reduction with W4A16.

### Our Recipe: GPTQModifier with W4A16

| Parameter | Value | Why |
|:--|:--|:--|
| `scheme` | `W4A16` | 4-bit weights  |
| `targets` | `Linear` | Linear layers hold most parameters - biggest savings |
| `ignore` | `["lm_head"]` | Output layer maps to vocabulary - keep it precise |

In [42]:
recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"],
)

print(f"Recipe: {recipe}")

Recipe: config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W4A16' kv_cache_scheme=None weight_observer=None input_observer=None output_observer=None observer=None bypass_divisibility_checks=False index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False block_size=128 dampening_frac=0.01 actorder=static offload_hessians=False


## Quantize the Model

### Why a calibration dataset?

GPTQ doesn't just round weights to lower precision, it uses a small set of real text to measure how each weight affects the model's output, then finds quantized values that minimize the error. This is what makes it more accurate than naive rounding.

The `dataset` parameter specifies what text to use for calibration. Here you'll use [WikiText-2](https://huggingface.co/datasets/mindchain/wikitext2), a standard benchmark of Wikipedia articles, the same dataset you'll use later for perplexity evaluation. You only need a few hundred samples as calibration is fast.

>**Note:** Since quantization can take several minutes and benefits from a GPU, we've already run it ahead of time and provided the quantized model in the `Qwen3-0.6B-W4A16` folder (`OUTPUT_DIR`). This learning environment is memory-constrained, so it might crash if you run the quantization yourself. The `if not os.path.isdir(OUTPUT_DIR)` check below ensures you skip re-running quantization when the folder already exists, so you can move straight to evaluation.


In [43]:
from llmcompressor import oneshot

if not os.path.isdir(OUTPUT_DIR):
    oneshot(
        model="Qwen/Qwen3-0.6B",
        dataset="wikitext",
        dataset_config_name="wikitext-2-raw-v1",
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=4096,
        num_calibration_samples=256,
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")

In [44]:
!python model_compare.py ../models/Qwen3-0.6B ../models/Qwen3-0.6B-W4A16

--> Hardware Detected: NVIDIA RTX PRO 4500 Blackwell (GPU Acceleration Enabled)

Model Size Comparison
Original (BF16):    1.41 GB
Quantized (W4A16):  524.4 MB
Reduction:          64%

Loading tokenizer from: ../models/Qwen3-0.6B
Loading Base model from ../models/Qwen3-0.6B...
Loading weights: 100%|█████████████████████| 311/311 [00:00<00:00, 1909.75it/s]

--- Base Model Response ---
Prompt: Machine learning is a branch of
Response:  artificial intelligence that has gained significant attention in recent years, particularly in the context of the rise of big data and the need for efficient, scalable solutions to complex problems. As the field continues to evolve, the integration of machine learning into various industries is becoming increasingly widespread. However, despite its growing popularity,

Loading Quantized model from ../models/Qwen3-0.6B-W4A16...
Decompressing model: 100%|█████████████████| 196/196 [00:00<00:00, 3037.92it/s]

--- Quantized Model Response ---
Prompt: Machine l